# 🔍 Notebook 2: Bias Detection & Measurement
## AI Bias & Fairness Detector — BCA Final Year Project

**Goal:** Quantify group-level disparities with industry-standard fairness metrics.

**Metrics:**
- **Demographic Parity Difference** — Should be < 0.10
- **Equalized Odds Difference** — Should be < 0.10  
- **Disparate Impact Ratio** — Should be ≥ 0.80
- **Selection Rate by Group** — Visualise per group

In [ ]:
import sys, os, warnings
sys.path.insert(0, r'C:\001-work\shree dive\Unbiased_AI_Decision')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from fairlearn.metrics import (
    demographic_parity_difference, equalized_odds_difference,
    MetricFrame, selection_rate
)
from src.preprocess import load_data, preprocess_data, get_train_test_splits

plt.style.use('dark_background')
BASE = r'C:\001-work\shree dive\Unbiased_AI_Decision'
CHARTS = os.path.join(BASE, 'outputs', 'charts')
os.makedirs(CHARTS, exist_ok=True)
print('Libraries loaded ✅')

## 1. Load Data & Baseline Model

In [ ]:
df = load_data()
X, y, sf = preprocess_data(df)
X_tr, X_te, y_tr, y_te, sf_tr, sf_te = get_train_test_splits(X, y, sf)

lr = joblib.load(os.path.join(BASE, 'outputs', 'models', 'lr_baseline.joblib'))
y_pred = lr.predict(X_te)

print(f'Test set size: {len(X_te)}')
print(f'Overall Accuracy: {accuracy_score(y_te, y_pred):.4f}')

## 2. Fairness Metrics Table

In [ ]:
dp_diff = demographic_parity_difference(y_te, y_pred, sensitive_features=sf_te['sex'])
eo_diff = equalized_odds_difference(y_te, y_pred, sensitive_features=sf_te['sex'])

mf_sel = MetricFrame(metrics=selection_rate, y_true=y_te, y_pred=y_pred, sensitive_features=sf_te['sex'])
sel_rates = mf_sel.by_group
dir_val = sel_rates.min() / sel_rates.max()

metrics_df = pd.DataFrame({
    'Metric': ['Demographic Parity Diff', 'Equalized Odds Diff', 'Disparate Impact Ratio'],
    'Value': [dp_diff, eo_diff, dir_val],
    'Threshold': ['< 0.10 (fair)', '< 0.10 (fair)', '>= 0.80 (fair)'],
    'Status': [
        '✅ FAIR' if abs(dp_diff) < 0.1 else '❌ BIASED',
        '✅ FAIR' if abs(eo_diff) < 0.1 else '❌ BIASED',
        '✅ FAIR' if dir_val >= 0.8 else '❌ BIASED',
    ]
})
print('=== BASELINE MODEL FAIRNESS METRICS (by Sex) ===')
print(metrics_df.to_string(index=False))
print(f'\nSelection Rate by Group:')
print(sel_rates.map('{:.1%}'.format))

## 3. Chart 2: Fairness Metric Dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Fairness Metric Dashboard — Baseline Model', fontsize=16, color='white', fontweight='bold')

metrics_names = ['Demographic Parity Diff', 'Equalized Odds Diff', 'Disparate Impact Ratio']
values = [abs(dp_diff), abs(eo_diff), dir_val]
thresholds = [0.10, 0.10, 0.80]
higher_better = [False, False, True]

for ax, name, val, thresh, hb in zip(axes, metrics_names, values, thresholds, higher_better):
    is_fair = (val < thresh) if not hb else (val >= thresh)
    color = '#2ECC71' if is_fair else '#E74C3C'
    bar = ax.bar([name.replace(' ', '\n')], [val], color=color, alpha=0.85, edgecolor='none', width=0.4)
    ax.axhline(thresh, color='#F39C12', linestyle='--', linewidth=2, label=f'Threshold: {thresh}')
    ax.set_ylim(0, max(val, thresh) * 1.4)
    ax.set_title(name, color='white', fontsize=11)
    ax.text(0, val + max(val, thresh)*0.05, f'{val:.4f}', ha='center', color='white', fontweight='bold', fontsize=13)
    ax.text(0, max(val, thresh)*1.25, '✅ FAIR' if is_fair else '❌ BIASED', ha='center',
            color='#2ECC71' if is_fair else '#E74C3C', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white')
    ax.legend(labelcolor='white', loc='upper right')
    ax.set_xticklabels([])

plt.tight_layout()
plt.savefig(os.path.join(CHARTS, 'fairness_metric_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved ✅')

## 4. Chart 4: Confusion Matrix Grid (Male vs Female)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Chart 4: Confusion Matrix Grid — Baseline Model (by Sex)', 
             fontsize=14, color='white', fontweight='bold')

for ax, group in zip(axes, ['Male', 'Female']):
    mask = sf_te['sex'] == group
    cm = confusion_matrix(y_te[mask], y_pred[mask])
    acc = accuracy_score(y_te[mask], y_pred[mask])
    
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, 
                cmap='Blues', linewidths=0.5,
                xticklabels=['<=50K', '>50K'],
                yticklabels=['<=50K', '>50K'])
    ax.set_title(f'{group} — Accuracy: {acc:.3f}', color='white', fontweight='bold')
    ax.set_xlabel('Predicted', color='white')
    ax.set_ylabel('Actual', color='white')
    ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig(os.path.join(CHARTS, 'confusion_matrix_grid.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix grid saved ✅')
print(f'\nAccuracy Gap (Male - Female): {accuracy_score(y_te[sf_te["sex"]=="Male"], y_pred[sf_te["sex"]=="Male"]) - accuracy_score(y_te[sf_te["sex"]=="Female"], y_pred[sf_te["sex"]=="Female"]):.4f}')

## 5. Summary

| Metric | Baseline Value | Fair Threshold | Result |
|---|---|---|---|
| Demographic Parity Diff | ~0.18 | < 0.10 | ❌ Biased |
| Equalized Odds Diff | ~0.16 | < 0.10 | ❌ Biased |
| Disparate Impact Ratio | ~0.29 | ≥ 0.80 | ❌ Biased |

> **Conclusion:** The baseline Logistic Regression model is heavily biased against females. This motivates the need for bias mitigation (see Notebook 3).